# 04 · Optimización RT-DETR Estados Unidos

Detector activo: **RT-DETRv2-S** con **OpenVINO Runtime nativo en Python**, checkpoint `merve/license-plates-rtdetrv2` con licencia Apache-2.0. Los datos y métricas nuevos se mantienen separados de los resultados históricos YOLO.


In [ ]:
%pip install -q "transformers>=5.4,<6" "accelerate>=1.2,<2" "pycocotools>=2.0.8" "optuna>=4,<5" "fast-plate-ocr==1.1.0" "roboflow==1.7.1" "openvino>=2025.4,<2027"

In [ ]:
from pathlib import Path
import os, subprocess, sys
REPO_ROOT=Path("/content/SIRAP_USA_ECUADOR")
if not REPO_ROOT.exists(): subprocess.run(["git","clone","--depth","1","https://github.com/eduardoceballos666/SIRAP_USA_ECUADOR.git",str(REPO_ROOT)],check=True)
os.chdir(REPO_ROOT); sys.path.insert(0,str(REPO_ROOT))
COUNTRY="usa"
MODEL_ID="merve/license-plates-rtdetrv2"
print("Perfil:",COUNTRY,"| RT-DETRv2-S:",MODEL_ID)


## Búsqueda de hiperparámetros
Los experimentos YOLO anteriores no se reutilizan como resultados RT-DETR. La búsqueda queda apagada hasta definir el dataset y presupuesto. Optimiza en validación y reserva test para la evaluación final.

In [ ]:
from pathlib import Path
import json
import optuna
from src.train import train_detector
from src.evaluate import evaluate_detector
DATA_DIR=None
RUN_SEARCH=False
OUTPUT=Path("/content/rtdetr_usa_optimizacion")

def objective(trial):
    lr=trial.suggest_float("learning_rate",1e-5,2e-4,log=True)
    wd=trial.suggest_float("weight_decay",1e-5,1e-2,log=True)
    out=OUTPUT/f"trial_{trial.number}/checkpoint"
    train_detector(DATA_DIR,"usa",MODEL_ID,out,out.parent/"trainer",epochs=12,batch_size=8,
                   learning_rate=lr,weight_decay=wd,seed=42,force=True)
    metrics=evaluate_detector(DATA_DIR,"usa","val",out,MODEL_ID,threshold=0.001,
        result_path=OUTPUT/f"trial_{trial.number}/metricas_val.json")
    return metrics["map50_95"] or 0.0
if RUN_SEARCH:
    if not DATA_DIR: raise ValueError("Define DATA_DIR con splits COCO")
    study=optuna.create_study(direction="maximize",study_name="sirap_rtdetr_usa")
    study.optimize(objective,n_trials=5)
    OUTPUT.mkdir(parents=True,exist_ok=True)
    study.trials_dataframe().to_csv(OUTPUT/"trials.csv",index=False)
    summary={"country":"usa","model_id":MODEL_ID,"objective":"map50_95_validation",
             "best_value":study.best_value,"best_params":study.best_trial.params,
             "n_trials":len(study.trials),"detector":"RT-DETRv2-S","inference_runtime":"OpenVINO Runtime"}
    (OUTPUT/"optimizacion_rtdetr_usa_resumen.json").write_text(json.dumps(summary,indent=2),encoding="utf-8")
    print(summary)
else: print("Búsqueda desactivada; define datos y presupuesto antes de activarla.")
